# Interacting Size-Consistent KQD — STO-3G & 6-31G

Multibasis notebook with optimisations from `krylov_h2_exact_sc_copy_4.ipynb`:

- Sweep caching (one Krylov chain per system at `max(N)`)
- `run_kqd_exact_dimer_fast` for Baseline 3
- Sparse `expm_multiply` for 6-31G joint Hamiltonians
- Sparse dressed product propagators for the novel procedure

**Quick smoke test:** `MULTIBASIS_SMOKE=1 jupyter nbconvert --execute krylov_interacting_sc_multibasis.ipynb`

**Regenerate notebook:** `python scripts/generate_multibasis_nb.py`

**Procedures:** (1) Novel N² dressed product, (2) separate fragments, (3) dimer fast vs H_tot.


In [1]:
# ── Imports and configuration ────────────────────────────────────────────────
import gc
import os
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy as sp
import scipy.linalg as la
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh, expm_multiply, norm as sparse_norm

from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.formats.molecule_info import MoleculeInfo
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock

FIG_ROOT = 'output/krylov_interacting_sc_multibasis'
os.makedirs(FIG_ROOT, exist_ok=True)
warnings.filterwarnings('ignore')

BOND = 0.735
R_LIST = [1000.0, 100.0, 10.0, 1.1]
SVD_THRESH = 1e-10
SPARSE_DIM_THRESHOLD = 256

FAST_MODE = os.environ.get('MULTIBASIS_SMOKE', '0') == '1'

BASIS_CONFIG = {
    'sto-3g': {
        'label': 'STO-3G',
        'N_LIST': [2, 3, 4, 5, 6],
        'N_NOVEL_EXT': list(range(2, 9)),
        'N_BASELINE_EXT': list(range(2, 17)),
    },
    '6-31g': {
        'label': '6-31G',
        'N_LIST': [2, 3, 4, 5, 6, 7, 8] if not FAST_MODE else [2, 3, 4],
        'N_NOVEL_EXT': list(range(2, 13)) if not FAST_MODE else [2, 3, 4],
        'N_BASELINE_EXT': list(range(2, 13)) if not FAST_MODE else [2, 3, 4],
    },
}

BASIS_LIST = ['sto-3g'] if FAST_MODE else ['sto-3g', '6-31g']
R_LIST_RUN = {b: ([1000.0] if FAST_MODE and b == '6-31g' else R_LIST) for b in BASIS_LIST}


def fig_dir(basis: str) -> str:
    d = os.path.join(FIG_ROOT, basis.replace('-', ''))
    os.makedirs(d, exist_ok=True)
    return d


print('Multibasis notebook ready')
print(f'  FIG_ROOT={FIG_ROOT}  FAST_MODE={FAST_MODE}')
print(f'  BASIS_LIST={BASIS_LIST}')


Multibasis notebook ready
  FIG_ROOT=output/krylov_interacting_sc_multibasis  FAST_MODE=False
  BASIS_LIST=['sto-3g', '6-31g']


In [2]:
# ── Dense KQD helpers (krylov_h2_exact_sc_copy_4.ipynb cell 5) ───────────────

def solve_gen_eig(H_tilde, S_tilde, threshold=SVD_THRESH):
    s_vals, s_vecs = sp.linalg.eigh(S_tilde)
    s_vecs = s_vecs.T
    good = np.array([v for val, v in zip(s_vals, s_vecs) if val > threshold])
    if len(good) == 0:
        return np.nan, 0
    H_reg = good.conj() @ H_tilde @ good.T
    S_reg = good.conj() @ S_tilde @ good.T
    return float(sp.linalg.eigh(H_reg, S_reg)[0][0].real), len(good)


def build_krylov_states(H_matrix, psi_ref, krylov_dim, dt):
    U = la.expm(-1j * H_matrix * dt)
    states = [psi_ref.copy().astype(complex)]
    for _ in range(1, krylov_dim):
        states.append(U @ states[-1])
    return states


def build_krylov_matrices_fast(H_matrix, states):
    Psi = np.column_stack(states)
    return Psi.conj().T @ Psi, Psi.conj().T @ (H_matrix @ Psi)


def run_kqd_exact(H_matrix, psi_ref, krylov_dim, dt, nuclear_rep, threshold=SVD_THRESH,
                  _states=None, _S_full=None, _H_full=None):
    if _states is None:
        _states = build_krylov_states(H_matrix, psi_ref, krylov_dim, dt)
        _S_full, _H_full = build_krylov_matrices_fast(H_matrix, _states)
    S_t, H_t = _S_full[:krylov_dim, :krylov_dim], _H_full[:krylov_dim, :krylov_dim]
    gs_elec, eff_dim = solve_gen_eig(H_t, S_t, threshold)
    s_evals = np.sort(np.linalg.eigvalsh(S_t).real)[::-1]
    return {
        'gs_elec': gs_elec, 'gs_total': gs_elec + nuclear_rep, 'eff_dim': eff_dim,
        'S': S_t, 's_evals': s_evals, 's_min': float(s_evals[-1]),
    }


def build_krylov_matrices_dimer_from_monomer(H_monomer, states_mono):
    S_m, H_m = build_krylov_matrices_fast(H_monomer, states_mono)
    return S_m * S_m, 2.0 * H_m * S_m, S_m, H_m


def run_kqd_exact_dimer_fast(H_monomer, psi_ref_monomer, krylov_dim, dt,
                              nuclear_rep_dimer, threshold=SVD_THRESH,
                              _states=None, _S_full=None, _H_full=None):
    if _states is None:
        _states = build_krylov_states(H_monomer, psi_ref_monomer, krylov_dim, dt)
        _S_full, _H_full, _, _ = build_krylov_matrices_dimer_from_monomer(H_monomer, _states)
    S_d, H_d = _S_full[:krylov_dim, :krylov_dim], _H_full[:krylov_dim, :krylov_dim]
    gs_elec, eff_dim = solve_gen_eig(H_d, S_d, threshold)
    s_evals = np.sort(np.linalg.eigvalsh(S_d).real)[::-1]
    return {
        'gs_elec': gs_elec, 'gs_total': gs_elec + nuclear_rep_dimer, 'eff_dim': eff_dim,
        'S': S_d, 's_evals': s_evals, 's_min': float(s_evals[-1]),
    }


def compute_dt(H_matrix):
    return np.pi / float(np.linalg.norm(H_matrix, ord=2))

print('Dense KQD helpers OK')


Dense KQD helpers OK


In [3]:
# ── Sparse KQD helpers (copy_4 cell 63) ──────────────────────────────────────

def build_sparse_hamiltonian(H_op):
    return H_op.to_matrix(sparse=True).tocsr()


def spectral_norm_sparse(H_sp):
    return float(sparse_norm(H_sp, ord=2))


def compute_dt_sparse(H_sp):
    return np.pi / spectral_norm_sparse(H_sp)


def build_krylov_states_sparse(H_sp, psi_ref, krylov_dim, dt):
    states = [psi_ref.copy().astype(complex)]
    for _ in range(1, krylov_dim):
        states.append(expm_multiply(-1j * dt * H_sp, states[-1]))
    return states


def build_krylov_projected_sparse(H_sp, states):
    Psi = np.column_stack(states)
    return Psi.conj().T @ Psi, Psi.conj().T @ (H_sp @ Psi)


def run_kqd_sparse(H_sp, psi_ref, krylov_dim, dt, E_nuc, threshold=SVD_THRESH,
                   _states=None, _S_full=None, _H_full=None):
    if _states is None:
        _states = build_krylov_states_sparse(H_sp, psi_ref, krylov_dim, dt)
        _S_full, _H_full = build_krylov_projected_sparse(H_sp, _states)
    S_t, H_t = _S_full[:krylov_dim, :krylov_dim], _H_full[:krylov_dim, :krylov_dim]
    gs_elec, eff_dim = solve_gen_eig(H_t, S_t, threshold)
    s_evals = np.sort(np.linalg.eigvalsh(S_t).real)[::-1]
    return {
        'gs_elec': gs_elec, 'gs_total': gs_elec + E_nuc, 'eff_dim': eff_dim,
        'S': S_t, 's_evals': s_evals, 's_min': float(s_evals[-1]),
    }


def exact_gs_electronic(H_dense=None, H_sp=None):
    if H_sp is not None:
        val, _ = eigsh(H_sp, k=1, which='SA')
        return float(val[0].real)
    return float(np.linalg.eigvalsh(H_dense)[0].real)

print('Sparse KQD helpers OK')


Sparse KQD helpers OK


## Step 1: Build systems per basis

In [4]:
# ── Build monomer + dimer + partition ─────────────────────────────────────────

def _pauli_support(label):
    n = len(label)
    return {n - 1 - j for j, ch in enumerate(label) if ch != 'I'}


def build_h2_monomer(basis, bond=BOND):
    half = bond / 2.0
    mol = MoleculeInfo(symbols=['H', 'H'], coords=[(0., 0., -half), (0., 0., half)],
                       charge=0, multiplicity=1)
    problem = PySCFDriver.from_molecule(mol, basis=basis).run()
    mapper = ParityMapper(num_particles=problem.num_particles)
    H_op = mapper.map(problem.hamiltonian.second_q_op())
    H = H_op.to_matrix().astype(complex)
    gs_elec = float(np.linalg.eigvalsh(H)[0].real)
    hf = HartreeFock(num_spatial_orbitals=problem.num_spatial_orbitals,
                     num_particles=problem.num_particles, qubit_mapper=mapper)
    ref = np.array(Statevector(hf.decompose()).data, dtype=complex)
    return {
        'H': H, 'H_op': H_op, 'ref': ref, 'E_nuc': problem.nuclear_repulsion_energy,
        'n_qubits': H_op.num_qubits, 'joint_dim': H.shape[0],
        'gs_elec': gs_elec, 'gs_total': gs_elec + problem.nuclear_repulsion_energy,
        'basis': basis,
    }


def build_h2_dimer(R_angstrom, basis, bond=BOND):
    b2 = bond / 2.0
    zA, zB = -R_angstrom / 2.0, R_angstrom / 2.0
    mol = MoleculeInfo(
        symbols=['H', 'H', 'H', 'H'],
        coords=[(0., 0., zA - b2), (0., 0., zA + b2), (0., 0., zB - b2), (0., 0., zB + b2)],
        charge=0, multiplicity=1,
    )
    problem = PySCFDriver.from_molecule(mol, basis=basis).run()
    mapper = ParityMapper(num_particles=problem.num_particles)
    H_op = mapper.map(problem.hamiltonian.second_q_op())
    joint_dim = 2 ** H_op.num_qubits
    use_sparse = joint_dim > SPARSE_DIM_THRESHOLD
    H_sp = build_sparse_hamiltonian(H_op) if use_sparse else None
    H = None if use_sparse else H_op.to_matrix().astype(complex)
    gs_elec = exact_gs_electronic(H_dense=H, H_sp=H_sp)
    hf = HartreeFock(num_spatial_orbitals=problem.num_spatial_orbitals,
                     num_particles=problem.num_particles, qubit_mapper=mapper)
    ref = np.array(Statevector(hf.decompose()).data, dtype=complex)
    return {
        'H': H, 'H_op': H_op, 'H_sp': H_sp, 'ref': ref,
        'E_nuc': problem.nuclear_repulsion_energy,
        'n_qubits': H_op.num_qubits, 'joint_dim': joint_dim,
        'gs_elec': gs_elec, 'gs_total': gs_elec + problem.nuclear_repulsion_energy,
        'R': R_angstrom, 'use_sparse': use_sparse, 'basis': basis,
    }


def partition_hamiltonian(H_op, qubits_a, qubits_b, use_sparse=False):
    qa, qb = set(qubits_a), set(qubits_b)
    pa, pb, pab, ca, cb, cab = [], [], [], [], [], []
    ab_sample = []
    for pauli, coeff in zip(H_op.paulis, H_op.coeffs):
        label = pauli.to_label()
        sup = _pauli_support(label)
        c = complex(coeff)
        if sup <= qa:
            pa.append(label); ca.append(c)
        elif sup <= qb:
            pb.append(label); cb.append(c)
        else:
            pab.append(label); cab.append(c)
            if len(ab_sample) < 5:
                ab_sample.append((label, c))

    def to_op(labels, coeffs):
        n = 2 ** H_op.num_qubits
        if not labels:
            return csr_matrix((n, n), dtype=complex) if use_sparse else np.zeros((n, n), dtype=complex)
        op = SparsePauliOp.from_list(list(zip(labels, coeffs)))
        return op.to_matrix(sparse=True).tocsr() if use_sparse else op.to_matrix().astype(complex)

    H_a, H_b, H_ab = to_op(pa, ca), to_op(pb, cb), to_op(pab, cab)
    if use_sparse:
        H_tot = H_op.to_matrix(sparse=True).tocsr()
        residual = spectral_norm_sparse(H_tot - (H_a + H_b + H_ab))
    else:
        H_tot = H_op.to_matrix().astype(complex)
        residual = float(np.linalg.norm(H_tot - H_a - H_b - H_ab, ord=2))
    return {
        'H_a': H_a, 'H_b': H_b, 'H_ab': H_ab, 'H_tot': H_tot,
        'n_a': len(pa), 'n_b': len(pb), 'n_ab': len(pab),
        'residual': residual, 'ab_sample': ab_sample,
    }


def build_dressed(H_a, H_b, H_ab):
    return H_a + 0.5 * H_ab, H_b + 0.5 * H_ab


def build_all_systems(basis, R_list):
    mono = build_h2_monomer(basis)
    qa = list(range(mono['n_qubits']))
    qb = list(range(mono['n_qubits'], 2 * mono['n_qubits']))
    systems, rows = {}, []
    print(f'\n=== {basis.upper()} | monomer dim={mono["joint_dim"]} ===')
    E_sum = 2.0 * mono['gs_total']
    for R in R_list:
        try:
            dim = build_h2_dimer(R, basis)
        except Exception as exc:
            print(f'  skip R={R}: {exc}')
            continue
        us = dim['use_sparse']
        part = partition_hamiltonian(dim['H_op'], qa, qb, use_sparse=us)
        H_A, H_B = build_dressed(part['H_a'], part['H_b'], part['H_ab'])
        if us:
            dress_res = spectral_norm_sparse(part['H_tot'] - H_A - H_B)
            norm_ab = spectral_norm_sparse(part['H_ab'])
            dt_tot = compute_dt_sparse(part['H_tot'])
        else:
            dress_res = float(np.linalg.norm(H_A + H_B - part['H_tot'], 2))
            norm_ab = float(np.linalg.norm(part['H_ab'], 2))
            dt_tot = compute_dt(part['H_tot'])
        sys = {
            **dim, **part, 'H_A': H_A, 'H_B': H_B,
            'E_exact_sum': E_sum, 'delta_exact': dim['gs_total'] - E_sum,
            'dress_residual': dress_res, 'dt_tot': dt_tot,
            'dt_mono': compute_dt(mono['H']), 'qubits_a': qa, 'qubits_b': qb,
        }
        if us:
            sys['H_tot_sp'] = part['H_tot']
            sys['H_A_sp'] = H_A
            sys['H_B_sp'] = H_B
        systems[R] = sys
        rows.append({
            'basis': basis, 'R': R, 'joint_dim': dim['joint_dim'], 'use_sparse': us,
            'E_exact_tot': dim['gs_total'], 'E_exact_sum': E_sum,
            'delta_exact': dim['gs_total'] - E_sum, 'norm_H_ab': norm_ab,
            'n_terms_ab': part['n_ab'], 'partition_residual': part['residual'],
            'dress_residual': dress_res,
        })
        print(f'  R={R:7.1f} A dim={dim["joint_dim"]:5d} sparse={us} GS={dim["gs_total"]:.8f} '
              f'delta={dim["gs_total"]-E_sum:.2e}')
        gc.collect()
    pd.DataFrame(rows).to_csv(f'{fig_dir(basis)}/step01_build.csv', index=False)
    return mono, systems


all_mono, all_systems = {}, {}
for basis in BASIS_LIST:
    all_mono[basis], all_systems[basis] = build_all_systems(basis, R_LIST_RUN[basis])



=== STO-3G | monomer dim=4 ===
  R= 1000.0 A dim=   64 sparse=False GS=-2.27461207 delta=1.60e-14
  R=  100.0 A dim=   64 sparse=False GS=-2.27461207 delta=2.04e-12
  R=   10.0 A dim=   64 sparse=False GS=-2.27461195 delta=1.24e-07
  R=    1.1 A dim=   64 sparse=False GS=-1.62751587 delta=6.47e-01

=== 6-31G | monomer dim=64 ===
  R= 1000.0 A dim=16384 sparse=True GS=-2.30322864 delta=-5.75e-10
  R=  100.0 A dim=16384 sparse=True GS=-2.30322864 delta=4.44e-12
  R=   10.0 A dim=16384 sparse=True GS=-2.30322866 delta=-2.40e-08
  R=    1.1 A dim=16384 sparse=True GS=-1.77847908 delta=5.25e-01


## Step 2: Optimised procedures

In [5]:
# ── Novel + baselines with caching ─────────────────────────────────────────────

def build_product_states_dense(H_A, H_B, ref, N, dt):
    U_A = la.expm(-1j * dt * H_A)
    U_B = la.expm(-1j * dt * H_B)
    b_chain = [ref.copy().astype(complex)]
    for _ in range(1, N):
        b_chain.append(U_B @ b_chain[-1])
    a_chain = [np.eye(H_A.shape[0], dtype=complex)]
    for _ in range(1, N):
        a_chain.append(U_A @ a_chain[-1])
    return [a_chain[i] @ b_chain[j] for i in range(N) for j in range(N)]


def build_product_states_sparse(H_A_sp, H_B_sp, ref, N, dt):
    ref = ref.astype(complex)
    b_chain = [ref.copy()]
    for _ in range(1, N):
        b_chain.append(expm_multiply(-1j * dt * H_B_sp, b_chain[-1]))
    states = []
    for i in range(N):
        for j in range(N):
            state = b_chain[j]
            for _ in range(i):
                state = expm_multiply(-1j * dt * H_A_sp, state)
            states.append(state)
    return states


def project_novel(sys, states, dim_n, threshold=SVD_THRESH):
    st = states[:dim_n]
    if sys['use_sparse']:
        S, Ht = build_krylov_projected_sparse(sys['H_tot_sp'], st)
    else:
        S, Ht = build_krylov_matrices_fast(sys['H_tot'], st)
    gs_elec, eff = solve_gen_eig(Ht, S, threshold)
    s_evals = np.sort(np.linalg.eigvalsh(S).real)[::-1]
    gs = gs_elec + sys['E_nuc']
    return {
        'gs_total': gs, 'gs_err': abs(gs - sys['gs_total']),
        'delta_sc': abs(gs - sys['E_exact_sum']), 'eff_dim': eff,
        's_min': float(s_evals[-1]), 'S': S,
    }


def run_kqd_joint(sys, N, threshold=SVD_THRESH, _st=None, _S=None, _H=None):
    if sys['use_sparse']:
        return run_kqd_sparse(sys['H_tot_sp'], sys['ref'], N, sys['dt_tot'], sys['E_nuc'],
                              threshold, _st, _S, _H)
    return run_kqd_exact(sys['H_tot'], sys['ref'], N, sys['dt_tot'], sys['E_nuc'],
                         threshold, _st, _S, _H)


def precompute(sys, mono, max_N):
    sm = build_krylov_states(mono['H'], mono['ref'], max_N, sys['dt_mono'])
    Sm, Hm = build_krylov_matrices_fast(mono['H'], sm)
    Sd, Hd, _, _ = build_krylov_matrices_dimer_from_monomer(mono['H'], sm)
    if sys['use_sparse']:
        sj = build_krylov_states_sparse(sys['H_tot_sp'], sys['ref'], max_N, sys['dt_tot'])
        Sj, Hj = build_krylov_projected_sparse(sys['H_tot_sp'], sj)
        sn = build_product_states_sparse(sys['H_A_sp'], sys['H_B_sp'], sys['ref'], max_N, sys['dt_tot'])
    else:
        sj = build_krylov_states(sys['H_tot'], sys['ref'], max_N, sys['dt_tot'])
        Sj, Hj = build_krylov_matrices_fast(sys['H_tot'], sj)
        sn = build_product_states_dense(sys['H_A'], sys['H_B'], sys['ref'], max_N, sys['dt_tot'])
    return {'mono': (sm, Sm, Hm), 'dimer': (sm, Sd, Hd), 'joint': (sj, Sj, Hj), 'novel': sn}


def sweep_all_fast(systems, mono, basis, N_list, threshold=SVD_THRESH):
    rows, max_N = [], max(N_list)
    for R, sys in sorted(systems.items()):
        c = precompute(sys, mono, max_N)
        sm, Sm, Hm = c['mono']
        _, Sd, Hd = c['dimer']
        sj, Sj, Hj = c['joint']
        sn = c['novel']
        for N in N_list:
            rn = project_novel(sys, sn, N * N, threshold)
            rows.append({
                'basis': basis, 'R': R, 'N': N, 'procedure': 'novel', 'basis_dim': N * N,
                **{k: rn[k] for k in ('gs_total', 'gs_err', 'delta_sc', 's_min', 'eff_dim')},
                'delta_int': np.nan,
                'E_exact_tot': sys['gs_total'], 'E_exact_sum': sys['E_exact_sum'],
                'delta_exact': sys['delta_exact'], 'joint_dim': sys['joint_dim'],
            })
            b2a = run_kqd_exact(mono['H'], mono['ref'], N, sys['dt_mono'], mono['E_nuc'],
                                threshold, sm, Sm, Hm)
            b2b = run_kqd_exact(mono['H'], mono['ref'], N, sys['dt_mono'], mono['E_nuc'],
                                threshold, sm, Sm, Hm)
            b2t = run_kqd_joint(sys, N, threshold, sj, Sj, Hj)
            rows.append({
                'basis': basis, 'R': R, 'N': N, 'procedure': 'baseline_separate', 'basis_dim': N,
                'gs_total': b2t['gs_total'], 'gs_err': abs(b2t['gs_total'] - sys['gs_total']),
                'delta_sc': abs(b2t['gs_total'] - sys['E_exact_sum']),
                'delta_int': abs(b2t['gs_total'] - b2a['gs_total'] - b2b['gs_total']),
                's_min': b2t['s_min'], 'eff_dim': b2t['eff_dim'],
                'E_a': b2a['gs_total'], 'E_b': b2b['gs_total'],
                'E_exact_tot': sys['gs_total'], 'E_exact_sum': sys['E_exact_sum'],
                'delta_exact': sys['delta_exact'], 'joint_dim': sys['joint_dim'],
            })
            bd = run_kqd_exact_dimer_fast(mono['H'], mono['ref'], N, sys['dt_mono'],
                                        2 * mono['E_nuc'], threshold, sm, Sd, Hd)
            bt = run_kqd_joint(sys, N, threshold, sj, Sj, Hj)
            rows.append({
                'basis': basis, 'R': R, 'N': N, 'procedure': 'baseline_dimer', 'basis_dim': N,
                'gs_total': bt['gs_total'], 'gs_err': abs(bt['gs_total'] - sys['gs_total']),
                'delta_sc': abs(bd['gs_total'] - sys['E_exact_sum']),
                'delta_int': abs(bt['gs_total'] - bd['gs_total']),
                's_min': bt['s_min'], 'eff_dim': bt['eff_dim'], 'E_dimer': bd['gs_total'],
                'E_exact_tot': sys['gs_total'], 'E_exact_sum': sys['E_exact_sum'],
                'delta_exact': sys['delta_exact'], 'joint_dim': sys['joint_dim'],
            })
    return pd.DataFrame(rows)

print('Procedures OK')


Procedures OK


## Step 3: Main sweep + plots (per basis)

In [6]:
# ── Sweep and plot per basis ───────────────────────────────────────────────────

PROC_STYLE = {
    'novel': ('-', 'C2', 'Novel (N² dressed)'),
    'baseline_separate': ('--', 'C0', 'Baseline 2: separate'),
    'baseline_dimer': (':', 'C1', 'Baseline 3: dimer fast'),
}

df_parts = []
for basis in BASIS_LIST:
    N_list = BASIS_CONFIG[basis]['N_LIST']
    print(f'\nSweeping {basis} ...')
    df_b = sweep_all_fast(all_systems[basis], all_mono[basis], basis, N_list)
    out = fig_dir(basis)
    df_b.to_csv(f'{out}/step02_sweep.csv', index=False)
    df_parts.append(df_b)
    systems = all_systems[basis]

    def plot_metric(metric, ylabel, fname, logy=True):
        fig, axes = plt.subplots(1, len(systems), figsize=(4 * len(systems), 4), sharey=True)
        if len(systems) == 1:
            axes = [axes]
        for ax, R in zip(axes, sorted(systems)):
            sub = df_b[df_b['R'] == R]
            for proc, (ls, c, lbl) in PROC_STYLE.items():
                s = sub[sub['procedure'] == proc].sort_values('N')
                y = np.maximum(s[metric].values, 1e-16) if logy else s[metric].values
                (ax.semilogy if logy else ax.plot)(s['N'], y, ls, color=c, marker='o', label=lbl)
            ax.set_title(f'R={R:.1f} Å'); ax.set_xlabel('N'); ax.grid(True, alpha=0.3)
            ax.legend(fontsize=7)
        axes[0].set_ylabel(ylabel)
        fig.suptitle(f'{BASIS_CONFIG[basis]["label"]}: {ylabel}', y=1.02)
        fig.tight_layout()
        fig.savefig(f'{out}/{fname}', dpi=150, bbox_inches='tight')
        plt.show()

    plot_metric('delta_sc', 'ΔE_SC [Ha]', 'sc_error_vs_N.png')
    plot_metric('gs_err', '|E − E_exact| [Ha]', 'gs_error_vs_N.png')
    print(df_b.groupby(['procedure', 'R'])['gs_err'].min())

df_sweep = pd.concat(df_parts, ignore_index=True)
df_sweep.to_csv(f'{FIG_ROOT}/step02_sweep_all.csv', index=False)
print(f'\nCombined sweep -> {FIG_ROOT}/step02_sweep_all.csv ({len(df_sweep)} rows)')



Sweeping sto-3g ...
procedure          R     
baseline_dimer     1.1       1.392447e-04
                   10.0      7.536194e-13
                   100.0     4.440892e-16
                   1000.0    0.000000e+00
baseline_separate  1.1       1.392447e-04
                   10.0      7.536194e-13
                   100.0     4.440892e-16
                   1000.0    0.000000e+00
novel              1.1       8.830740e-07
                   10.0      1.854636e-10
                   100.0     8.881784e-16
                   1000.0    0.000000e+00
Name: gs_err, dtype: float64

Sweeping 6-31g ...
procedure          R     
baseline_dimer     1.1       0.000635
                   10.0      0.000001
                   100.0     0.000001
                   1000.0    0.000001
baseline_separate  1.1       0.000635
                   10.0      0.000001
                   100.0     0.000001
                   1000.0    0.000001
novel              1.1       0.005575
                   10.0      0.0

## Step 3a: Energy separation R=1.1 vs R=1000 (per basis)

In [7]:
# ── Energy separation |E(1.1) - E(1000)| vs N ─────────────────────────────────
R_CLOSE, R_REF = 1.1, 1000.0

for basis in BASIS_LIST:
    systems = all_systems[basis]
    if R_CLOSE not in systems or R_REF not in systems:
        continue
    out = fig_dir(basis)
    N_list = BASIS_CONFIG[basis]['N_LIST']
    sub = df_sweep[df_sweep['basis'] == basis]
    delta_exact = abs(systems[R_CLOSE]['gs_total'] - systems[R_REF]['gs_total'])
    rows = []
    for proc in PROC_STYLE:
        for N in N_list:
            g1 = sub[(sub['R']==R_CLOSE)&(sub['procedure']==proc)&(sub['N']==N)]['gs_total']
            g0 = sub[(sub['R']==R_REF)&(sub['procedure']==proc)&(sub['N']==N)]['gs_total']
            if len(g1)==0 or len(g0)==0:
                continue
            dsep = abs(float(g1.iloc[0]) - float(g0.iloc[0]))
            rows.append({'N': N, 'procedure': proc, 'delta_sep': dsep,
                         'sep_err': abs(dsep - delta_exact)})
    df_sep = pd.DataFrame(rows)
    df_sep.to_csv(f'{out}/step03a_energy_sep.csv', index=False)
    fig, ax = plt.subplots(figsize=(8, 5))
    for proc, (ls, c, lbl) in PROC_STYLE.items():
        s = df_sep[df_sep['procedure']==proc].sort_values('N')
        ax.plot(s['N'], s['delta_sep'], ls, color=c, marker='o', label=lbl)
    ax.axhline(delta_exact, color='k', ls='--', label=f'exact = {delta_exact:.4f} Ha')
    ax.set_xlabel('N'); ax.set_ylabel('ΔE_sep [Ha]')
    ax.set_title(f'{BASIS_CONFIG[basis]["label"]}: energy separation')
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(f'{out}/energy_sep_R1p1_R1000_vs_N.png', dpi=150, bbox_inches='tight')
    plt.show()


## Step 3d: Interaction energy vs N and R

In [8]:
# ── Interaction energy E_int(R) = E(R) - E(R_ref) ─────────────────────────────
R_REF = 1000.0

for basis in BASIS_LIST:
    systems = all_systems[basis]
    if R_REF not in systems:
        continue
    out = fig_dir(basis)
    N_list = BASIS_CONFIG[basis]['N_LIST']
    sub = df_sweep[df_sweep['basis'] == basis]
    E_ref = systems[R_REF]['gs_total']
    rows = []
    for R in systems:
        E_int_ex = systems[R]['gs_total'] - E_ref
        for proc in PROC_STYLE:
            for N in N_list:
                gR = sub[(sub['R']==R)&(sub['procedure']==proc)&(sub['N']==N)]['gs_total']
                g0 = sub[(sub['R']==R_REF)&(sub['procedure']==proc)&(sub['N']==N)]['gs_total']
                if len(gR)==0 or len(g0)==0:
                    continue
                E_int = float(gR.iloc[0] - g0.iloc[0])
                rows.append({'R': R, 'N': N, 'procedure': proc,
                             'E_int': E_int, 'E_int_exact': E_int_ex,
                             'int_err': abs(E_int - E_int_ex)})
    df_int = pd.DataFrame(rows)
    df_int.to_csv(f'{out}/step3d_int_energy.csv', index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, Rf in zip(axes, [10.0, 1.1]):
        if Rf not in systems:
            continue
        E_int_ex = systems[Rf]['gs_total'] - E_ref
        for proc, (ls, c, lbl) in PROC_STYLE.items():
            s = df_int[(df_int['R']==Rf)&(df_int['procedure']==proc)].sort_values('N')
            ax.semilogy(s['N'], np.maximum(s['int_err'], 1e-16), ls, color=c, marker='o', label=lbl)
        ax.set_title(f'R={Rf} Å  exact E_int={E_int_ex:.3e} Ha')
        ax.set_xlabel('N'); ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('|E_int − E_int_exact| [Ha]')
    axes[0].legend(fontsize=8)
    fig.suptitle(f'{BASIS_CONFIG[basis]["label"]}: interaction energy error')
    fig.tight_layout()
    fig.savefig(f'{out}/int_energy_error_vs_N.png', dpi=150, bbox_inches='tight')
    plt.show()


## Step 3c: Extended GS convergence at R = 1.1 Å

In [9]:
# ── Extended GS convergence (per basis) ───────────────────────────────────────

R_GS = 1.1
CHEM_ACC = 1.6e-3
GS_STYLE = {
    'novel': ('-', 'C2', 'o', 'Novel N²'),
    'baseline_separate': ('--', 'C0', 's', 'Baseline 2'),
    'baseline_dimer': (':', 'C1', '^', 'Baseline 3'),
}

for basis in BASIS_LIST:
    if R_GS not in all_systems[basis]:
        print(f'skip 3c {basis}: R={R_GS} missing')
        continue
    cfg = BASIS_CONFIG[basis]
    sys_gs = all_systems[basis][R_GS]
    mono = all_mono[basis]
    out = fig_dir(basis)
    JOINT_DIM = sys_gs['joint_dim']
    max_N = max(max(cfg['N_NOVEL_EXT']), max(cfg['N_BASELINE_EXT']))
    cache = precompute(sys_gs, mono, max_N)
    sm, Sm, Hm = cache['mono']
    _, Sd, Hd = cache['dimer']
    sj, Sj, Hj = cache['joint']
    sn = cache['novel']
    rows = []
    for N in cfg['N_NOVEL_EXT']:
        rn = project_novel(sys_gs, sn, N * N)
        rows.append({'basis': basis, 'R': R_GS, 'N': N, 'procedure': 'novel',
                     'basis_dim': N*N, **{k: rn[k] for k in ('eff_dim','gs_total','gs_err','s_min')}})
    for N in cfg['N_BASELINE_EXT']:
        bt = run_kqd_joint(sys_gs, N, _st=sj, _S=Sj, _H=Hj)
        rows.append({'basis': basis, 'R': R_GS, 'N': N, 'procedure': 'baseline_separate',
                     'basis_dim': N, 'eff_dim': bt['eff_dim'], 'gs_total': bt['gs_total'],
                     'gs_err': abs(bt['gs_total']-sys_gs['gs_total']), 's_min': bt['s_min']})
        rows.append({'basis': basis, 'R': R_GS, 'N': N, 'procedure': 'baseline_dimer',
                     'basis_dim': N, 'eff_dim': bt['eff_dim'], 'gs_total': bt['gs_total'],
                     'gs_err': abs(bt['gs_total']-sys_gs['gs_total']), 's_min': bt['s_min']})
    df_gs = pd.DataFrame(rows)
    df_gs.to_csv(f'{out}/step3c_gs_convergence_R1p1.csv', index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    for proc, (ls, c, mk, lbl) in GS_STYLE.items():
        s = df_gs[df_gs['procedure']==proc].sort_values('basis_dim')
        ax.semilogy(s['basis_dim'], np.maximum(s['gs_err'], 1e-16), ls, color=c, marker=mk, label=lbl)
    ax.axhline(CHEM_ACC, color='gray', ls=':', label='1.6 mHa')
    ax.set_xlabel('Basis size'); ax.set_ylabel('|E − E_exact| [Ha]')
    ax.set_title(f'{cfg["label"]} GS convergence at R={R_GS} Å (dim={JOINT_DIM})')
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(f'{out}/gs_error_convergence_R1p1.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}/gs_error_convergence_R1p1.png')


Saved output/krylov_interacting_sc_multibasis/sto3g/gs_error_convergence_R1p1.png
Saved output/krylov_interacting_sc_multibasis/631g/gs_error_convergence_R1p1.png


## Step 4: Summary

In [10]:
# ── Summary table ──────────────────────────────────────────────────────────────
for basis in BASIS_LIST:
    print(f'\n=== {basis.upper()} summary (N=4 or max N_LIST) ===')
    N_pick = max(BASIS_CONFIG[basis]['N_LIST'])
    sub = df_sweep[(df_sweep['basis']==basis) & (df_sweep['N']==N_pick)]
    print(sub.groupby('R')[['procedure','gs_err','delta_sc']].apply(
        lambda g: g.set_index('procedure')[['gs_err','delta_sc']].to_string()).to_string())
print(f'\nOutputs under {FIG_ROOT}/')



=== STO-3G summary (N=4 or max N_LIST) ===
R
1.1                                gs_err      delta_sc\...
10.0                               gs_err      delta_sc\...
100.0                              gs_err      delta_sc\...
1000.0                             gs_err      delta_sc\...

=== 6-31G summary (N=4 or max N_LIST) ===
R
1.1                            gs_err      delta_sc\npro...
10.0                           gs_err      delta_sc\npro...
100.0                          gs_err      delta_sc\npro...
1000.0                         gs_err      delta_sc\npro...

Outputs under output/krylov_interacting_sc_multibasis/
